### Import libraries and load datasets

In [93]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FEATURES_DIR = ROOT / "data" / "interim" / "tfidf_features"
MODEL_DIR = ROOT / "models" / "traditional_ml"
OUTPUT_DIR = ROOT / "outputs" / "evaluation"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)

Project root: d:\My_Project\Analyse\mobile-review-sentiment-analysis


### Load TF-IDF features and labels

In [94]:
X_train = load_npz(FEATURES_DIR / "X_train_tfidf.npz")
X_val = load_npz(FEATURES_DIR / "X_validation_tfidf.npz")
X_test = load_npz(FEATURES_DIR / "X_test_tfidf.npz")

y_train = pd.read_csv(FEATURES_DIR / "y_train.csv")["sentiment"].astype(str).str.strip().str.lower()
y_val = pd.read_csv(FEATURES_DIR / "y_validation.csv")["sentiment"].astype(str).str.strip().str.lower()
y_test = pd.read_csv(FEATURES_DIR / "y_test.csv")["sentiment"].astype(str).str.strip().str.lower()

assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)
assert X_test.shape[0] == len(y_test)

assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

print("Data validation passed.")

Data validation passed.


### Check sentiment distribution

In [95]:
print("Training sentiment distribution:")
print(y_train.value_counts())

print("\nValidation sentiment distribution:")
print(y_val.value_counts())

Training sentiment distribution:
sentiment
positive    11475
negative     2686
neutral      1830
Name: count, dtype: int64

Validation sentiment distribution:
sentiment
positive    1435
negative     336
neutral      228
Name: count, dtype: int64


### Balance training data with random oversampling

In [96]:
rng = np.random.default_rng(42)
y_train_values = y_train.to_numpy()
target_count = int(y_train.value_counts().max())
balanced_indices = np.concatenate([
    rng.choice(
        np.flatnonzero(y_train_values == label),
        size=target_count,
        replace=np.count_nonzero(y_train_values == label) < target_count
    )
    for label in y_train.unique()
])
rng.shuffle(balanced_indices)

X_train_balanced = X_train[balanced_indices]
y_train_balanced = y_train.iloc[balanced_indices].reset_index(drop=True)

print("Balanced training sentiment distribution:")
print(y_train_balanced.value_counts())

Balanced training sentiment distribution:
sentiment
negative    11475
positive    11475
neutral     11475
Name: count, dtype: int64


### Tune model hyperparameters and balance strategy

In [97]:
c_values = [0.5, 1.0, 2.0, 5.0]
balance_strategies = ["Original", "Class-weighted", "Oversampled"]

In [98]:
trained_models = {}
validation_rows = []

for model_type in ("Logistic Regression", "Linear SVM"):
    for c_value in c_values:
        for balance_strategy in balance_strategies:
            model_name = f"{model_type} (C={c_value}, {balance_strategy})"
            print(f"Training {model_name}...")

            if balance_strategy == "Oversampled":
                fit_X, fit_y = X_train_balanced, y_train_balanced
                class_weight = None
            else:
                fit_X, fit_y = X_train, y_train
                class_weight = (
                    "balanced" if balance_strategy == "Class-weighted" else None
                )

            if model_type == "Logistic Regression":
                classifier = LogisticRegression(
                    C=c_value,
                    max_iter=2000,
                    class_weight=class_weight,
                    random_state=42
                )
            else:
                classifier = LinearSVC(
                    C=c_value,
                    class_weight=class_weight,
                    random_state=42
                )

            classifier.fit(fit_X, fit_y)
            trained_models[model_name] = classifier
            predictions = classifier.predict(X_val)
            precision, recall, macro_f1, _ = precision_recall_fscore_support(
                y_val, predictions, average="macro", zero_division=0
            )
            validation_rows.append({
                "Model": model_name,
                "Balance": balance_strategy,
                "C": c_value,
                "Accuracy": accuracy_score(y_val, predictions),
                "Macro Precision": precision,
                "Macro Recall": recall,
                "Macro F1": macro_f1,
                "Decision Bias": "Default"
            })

results_df = (
    pd.DataFrame(validation_rows)
    .sort_values(["Macro F1", "Accuracy"], ascending=False)
    .reset_index(drop=True)
)
display(results_df)


Training Logistic Regression (C=0.5, Original)...
Training Logistic Regression (C=0.5, Class-weighted)...
Training Logistic Regression (C=0.5, Oversampled)...
Training Logistic Regression (C=1.0, Original)...
Training Logistic Regression (C=1.0, Class-weighted)...
Training Logistic Regression (C=1.0, Oversampled)...
Training Logistic Regression (C=2.0, Original)...
Training Logistic Regression (C=2.0, Class-weighted)...
Training Logistic Regression (C=2.0, Oversampled)...
Training Logistic Regression (C=5.0, Original)...
Training Logistic Regression (C=5.0, Class-weighted)...
Training Logistic Regression (C=5.0, Oversampled)...
Training Linear SVM (C=0.5, Original)...
Training Linear SVM (C=0.5, Class-weighted)...
Training Linear SVM (C=0.5, Oversampled)...
Training Linear SVM (C=1.0, Original)...
Training Linear SVM (C=1.0, Class-weighted)...
Training Linear SVM (C=1.0, Oversampled)...
Training Linear SVM (C=2.0, Original)...
Training Linear SVM (C=2.0, Class-weighted)...
Training Lin

,Model,Balance,C,Accuracy,Macro Precision,Macro Recall,Macro F1,Decision Bias
0,"Logistic Regression (C=2.0, Oversampled)",Oversampled,2.0,0.796398,0.653524,0.660253,0.656809,Default
1,"Logistic Regression (C=0.5, Oversampled)",Oversampled,0.5,0.781391,0.642452,0.673899,0.655495,Default
2,"Logistic Regression (C=1.0, Class-weighted)",Class-weighted,1.0,0.780390,0.639288,0.675823,0.654395,Default
3,"Logistic Regression (C=5.0, Oversampled)",Oversampled,5.0,0.800900,0.657738,0.650226,0.653830,Default
4,"Logistic Regression (C=2.0, Class-weighted)",Class-weighted,2.0,0.785893,0.641457,0.665032,0.652038,Default
5,"Logistic Regression (C=5.0, Class-weighted)",Class-weighted,5.0,0.792896,0.645677,0.654757,0.650062,Default
6,"Logistic Regression (C=0.5, Class-weighted)",Class-weighted,0.5,0.766883,0.630823,0.683838,0.650019,Default
7,"Linear SVM (C=0.5, Oversampled)",Oversampled,0.5,0.797399,0.649099,0.642162,0.645333,Default
8,"Logistic Regression (C=1.0, Oversampled)",Oversampled,1.0,0.783392,0.637035,0.654683,0.644984,Default
9,"Linear SVM (C=1.0, Class-weighted)",Class-weighted,1.0,0.807404,0.661308,0.632161,0.642913,Default


### Calibrate and save the best model

In [99]:
best_name = results_df.loc[0, "Model"]
best_model = trained_models[best_name]

decision_scores = best_model.decision_function(X_val)
if decision_scores.ndim != 2:
    raise ValueError("Expected multiclass decision scores for sentiment labels")

model_classes = np.asarray(best_model.classes_)
bias_range = np.arange(-0.5, 0.5001, 0.025)
calibration_rows = []

for negative_bias in bias_range:
    for neutral_bias in bias_range:
        class_bias = np.zeros(len(model_classes))
        class_bias[model_classes == "negative"] = negative_bias
        class_bias[model_classes == "neutral"] = neutral_bias
        adjusted_predictions = model_classes[
            np.argmax(decision_scores + class_bias, axis=1)
        ]
        precision, recall, macro_f1, _ = precision_recall_fscore_support(
            y_val, adjusted_predictions, average="macro", zero_division=0
        )
        calibration_rows.append({
            "Bias": class_bias,
            "Accuracy": accuracy_score(y_val, adjusted_predictions),
            "Macro Precision": precision,
            "Macro Recall": recall,
            "Macro F1": macro_f1,
            "Negative Bias": negative_bias,
            "Neutral Bias": neutral_bias
        })

best_calibration = max(
    calibration_rows,
    key=lambda row: (row["Macro F1"], row["Accuracy"])
)
best_model.intercept_ = best_model.intercept_ + best_calibration["Bias"]
trained_models[best_name] = best_model

best_result_index = results_df.index[results_df["Model"] == best_name][0]
for metric in ("Accuracy", "Macro Precision", "Macro Recall", "Macro F1"):
    results_df.loc[best_result_index, metric] = best_calibration[metric]
results_df.loc[best_result_index, "Decision Bias"] = (
    f"negative={best_calibration['Negative Bias']:.3f}, "
    f"neutral={best_calibration['Neutral Bias']:.3f}"
)
results_df = (
    results_df.sort_values(["Macro F1", "Accuracy"], ascending=False)
    .reset_index(drop=True)
)
display(results_df.head(10))

joblib.dump(
    best_model,
    MODEL_DIR / "best_tfidf_model.joblib"
)

print("Best model:", best_name)
print("Validation metrics after decision-bias tuning:")
print({metric: best_calibration[metric] for metric in (
    "Accuracy", "Macro Precision", "Macro Recall", "Macro F1"
)})

,Model,Balance,C,Accuracy,Macro Precision,Macro Recall,Macro F1,Decision Bias
0,"Logistic Regression (C=2.0, Oversampled)",Oversampled,2.0,0.793397,0.656707,0.668297,0.661304,"negative=-0.050, neutral=0.125"
1,"Logistic Regression (C=0.5, Oversampled)",Oversampled,0.5,0.781391,0.642452,0.673899,0.655495,Default
2,"Logistic Regression (C=1.0, Class-weighted)",Class-weighted,1.0,0.780390,0.639288,0.675823,0.654395,Default
3,"Logistic Regression (C=5.0, Oversampled)",Oversampled,5.0,0.800900,0.657738,0.650226,0.653830,Default
4,"Logistic Regression (C=2.0, Class-weighted)",Class-weighted,2.0,0.785893,0.641457,0.665032,0.652038,Default
5,"Logistic Regression (C=5.0, Class-weighted)",Class-weighted,5.0,0.792896,0.645677,0.654757,0.650062,Default
6,"Logistic Regression (C=0.5, Class-weighted)",Class-weighted,0.5,0.766883,0.630823,0.683838,0.650019,Default
7,"Linear SVM (C=0.5, Oversampled)",Oversampled,0.5,0.797399,0.649099,0.642162,0.645333,Default
8,"Logistic Regression (C=1.0, Oversampled)",Oversampled,1.0,0.783392,0.637035,0.654683,0.644984,Default
9,"Linear SVM (C=1.0, Class-weighted)",Class-weighted,1.0,0.807404,0.661308,0.632161,0.642913,Default


Best model: Logistic Regression (C=2.0, Oversampled)
Validation metrics after decision-bias tuning:
{'Accuracy': 0.7933966983491746, 'Macro Precision': 0.6567074480818854, 'Macro Recall': 0.6682967581555515, 'Macro F1': 0.6613037397732132}


### Save Logistic Regression separately

In [100]:
logistic_candidates = results_df[
    results_df["Model"].str.startswith("Logistic Regression")
]
logistic_name = logistic_candidates.iloc[0]["Model"]
logistic_model = trained_models[logistic_name]
joblib.dump(
    logistic_model,
    MODEL_DIR / "logistic_regression_tfidf.joblib"
)

['d:\\My_Project\\Analyse\\mobile-review-sentiment-analysis\\models\\traditional_ml\\logistic_regression_tfidf.joblib']

### Save validation results

In [101]:
results_path = OUTPUT_DIR / "traditional_validation_results.csv"

results_df.to_csv(
    results_path,
    index=False
)

print("Results saved to:", results_path)

Results saved to: d:\My_Project\Analyse\mobile-review-sentiment-analysis\outputs\evaluation\traditional_validation_results.csv


### Classification report

In [102]:
val_predictions = best_model.predict(X_val)

print("Best model:", best_name)

print("\nClassification Report:")

print(
    classification_report(
        y_val,
        val_predictions,
        zero_division=0
    )
)

Best model: Logistic Regression (C=2.0, Oversampled)

Classification Report:
              precision    recall  f1-score   support

    negative       0.70      0.68      0.69       336
     neutral       0.38      0.45      0.41       228
    positive       0.90      0.87      0.89      1435

    accuracy                           0.79      1999
   macro avg       0.66      0.67      0.66      1999
weighted avg       0.80      0.79      0.80      1999



### Confusion matrix

In [103]:
labels = sorted(y_val.unique())

cm = confusion_matrix(
    y_val,
    val_predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in labels],
    columns=[f"Predicted {label}" for label in labels]
)

display(cm_df)

,Predicted negative,Predicted neutral,Predicted positive
Actual negative,228,53,55
Actual neutral,37,103,88
Actual positive,63,117,1255
